# 04. HS 분류 지식베이스 검색 및 근거 결합

03번의 TF-IDF + Linear SVM을 기준 모델로 두고, 101개 HS 분류 지식베이스에서 제품별 관련 규칙을 검색한다. 이번 단계는 RAG의 검색(Retrieval) 파이프라인을 구축하고, 예측 코드와 검색 근거의 연결성을 평가하는 단계다.

주의: 검색된 문서가 정답 HS 코드를 자동으로 확정하거나 수정하지 않는다. 특히 `hs_code`가 비어 있는 공통규정 문서는 코드별 근거로 단정하지 않고, 적용범위와 문서 내용을 함께 제시한다. 실제 코드 변경은 규칙 검토 및 검증을 거쳐야 한다.

In [ ]:
!pip -q install pandas numpy scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import json
import ast
from pathlib import Path
from urllib.parse import quote
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

BASE = 'https://raw.githubusercontent.com/lin1118-code/HS-complete-product/main/'
DATA_FILE = '완제품 모델 데이터_사례_전처리 후 - 사례_전처리 완료.csv'
KB_FILE = 'hs_knowledge_base.jsonl'
df = pd.read_csv(BASE + quote(DATA_FILE), encoding='utf-8-sig').fillna('')
kb = pd.read_json(BASE + KB_FILE, lines=True).fillna('')
INPUT_COLUMNS = ['Product Name', 'Main Ingredients (KR)', 'Main Ingredients (EN)', 'Product Form and Packaging', 'Intended Use']
TARGET_COLUMN = 'HS Code'
for col in INPUT_COLUMNS + [TARGET_COLUMN]: df[col] = df[col].astype(str).str.strip()
df = df[df[TARGET_COLUMN].ne('')].reset_index(drop=True)
df['model_text'] = '[PRODUCT] ' + df['Product Name'] + ' [INGREDIENTS_KR] ' + df['Main Ingredients (KR)'] + ' [INGREDIENTS_EN] ' + df['Main Ingredients (EN)'] + ' [FORM] ' + df['Product Form and Packaging'] + ' [USE] ' + df['Intended Use']
df['duplicate_group'] = pd.util.hash_pandas_object(df[INPUT_COLUMNS], index=False).astype(str)
print('사례 데이터:', len(df), '건 /', df[TARGET_COLUMN].nunique(), '개 HS 코드')
print('지식베이스:', len(kb), '건')
display(kb.groupby(['source_sheet','rule_type']).size().rename('count').reset_index())

## 1. 지식베이스 구조 점검

문서별 적용범위, 규칙유형, HS 코드, 원문을 확인한다. HS 코드가 명시되지 않은 문서도 제33류 공통규정처럼 중요한 범위 규칙일 수 있으므로 삭제하지 않는다.

In [ ]:
required = ['knowledge_id','source_sheet','hs_code','rule_type','document_text']
missing = [c for c in required if c not in kb.columns]
if missing: raise ValueError(f'지식베이스 필수 열 누락: {missing}')
for col in required: kb[col] = kb[col].astype(str).fillna('').str.strip()
kb['search_text'] = kb['document_text'] + ' ' + kb['source_sheet'] + ' ' + kb['rule_type'] + ' ' + kb['hs_code']
kb['chapter33_related'] = kb['document_text'].str.contains('제33류|제33류|제33장|제330[1-7]호|330[1-7]호', regex=True)
print('문서 수:', len(kb), '| 고유 ID:', kb['knowledge_id'].nunique())
print('HS 코드 직접 명시 문서:', kb['hs_code'].ne('').sum())
print('제33류 관련 텍스트 문서:', kb['chapter33_related'].sum())
display(kb[['knowledge_id','source_sheet','hs_code','rule_type','document_text']].head(10))

## 2. 제품 설명 → 지식베이스 검색

단어 n-gram과 문자 n-gram을 결합한 TF-IDF 검색기를 지식베이스 전체 문서에 적합한다. 각 제품에 대해 유사한 규칙 문서 Top-K를 반환한다. 이 검색기는 의미를 이해하는 임베딩 모델이 아니라 어휘·문자 패턴 기반 검색 기준선이다.

In [ ]:
def make_vectorizer():
    return FeatureUnion([
        ('word', TfidfVectorizer(analyzer='word', ngram_range=(1,2), min_df=1, max_features=40000, sublinear_tf=True)),
        ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5), min_df=1, max_features=40000, sublinear_tf=True))
    ])

kb_vectorizer = make_vectorizer()
kb_matrix = kb_vectorizer.fit_transform(kb['search_text'])

def retrieve_rules(query, top_k=5):
    q_vec = kb_vectorizer.transform([query])
    sims = cosine_similarity(q_vec, kb_matrix).ravel()
    order = np.argsort(sims)[::-1][:top_k]
    out = kb.iloc[order][['knowledge_id','source_sheet','hs_code','rule_type','source_page_or_line','document_text']].copy()
    out['similarity'] = sims[order]
    return out.reset_index(drop=True)

sample_query = df.loc[0, 'model_text']
display(retrieve_rules(sample_query, top_k=5))

## 3. Linear SVM 예측 + 규칙 검색 결과 결합

각 Fold에서 Linear SVM을 학습하고, 테스트 제품별 예측 코드와 상위 3개 후보를 만든다. 별도로 지식베이스 Top-5 문서를 검색해 저장한다. 우선은 분류기 결과를 유지하고 근거를 덧붙이는 보수적 결합 방식으로 구현한다.

In [ ]:
class_counts = df[TARGET_COLUMN].value_counts()
eligible = class_counts[class_counts >= 4].index
eval_df = df[df[TARGET_COLUMN].isin(eligible)].copy()
group_counts = eval_df.groupby(TARGET_COLUMN)['duplicate_group'].nunique()
eligible = group_counts[group_counts >= 4].index
eval_df = eval_df[eval_df[TARGET_COLUMN].isin(eligible)].reset_index(drop=True)
X = eval_df['model_text'].reset_index(drop=True)
y = eval_df[TARGET_COLUMN].reset_index(drop=True)
groups = eval_df['duplicate_group'].reset_index(drop=True)
labels = np.sort(y.unique())
cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
splits = list(cv.split(X, y, groups))

pred_rows, evidence_rows = [], []
for fold, (tr, te) in enumerate(splits, 1):
    vec = make_vectorizer()
    xtr, xte = vec.fit_transform(X.iloc[tr]), vec.transform(X.iloc[te])
    model = LinearSVC(C=1.0, class_weight='balanced').fit(xtr, y.iloc[tr])
    scores = model.decision_function(xte)
    for j, idx in enumerate(te):
        ranked = np.argsort(scores[j])[::-1]
        candidates = [model.classes_[i] for i in ranked[:3]]
        pred_rows.append({'fold':fold,'row_index':int(idx),'true_hs':y.iloc[idx],'predicted_hs':candidates[0],'top3_candidates':candidates})
        rules = retrieve_rules(X.iloc[idx], top_k=5)
        for rank, r in rules.iterrows():
            evidence_rows.append({'fold':fold,'row_index':int(idx),'true_hs':y.iloc[idx],'predicted_hs':candidates[0],'candidate_hs_top3':candidates,'rank':int(rank+1),'knowledge_id':r['knowledge_id'],'source_sheet':r['source_sheet'],'rule_hs_code':r['hs_code'],'rule_type':r['rule_type'],'source_page_or_line':r['source_page_or_line'],'similarity':float(r['similarity']),'document_text':r['document_text']})

pred_df = pd.DataFrame(pred_rows)
evidence_df = pd.DataFrame(evidence_rows)
print('테스트 예측:', len(pred_df), '| 검색 근거:', len(evidence_df))
display(pred_df.head())
display(evidence_df.head(10))

## 4. 지식베이스 검색의 유효성 점검

현재 지식베이스 문서 중 상당수는 HS 코드가 직접 연결되지 않은 공통규정·분류기준일 수 있다. 따라서 검색 결과의 HS 코드 일치율만으로 검색 품질을 평가하지 않는다. 아래에서는 (1) 검색된 문서의 유형, (2) 직접 코드가 있는 문서의 후보 코드 일치 여부, (3) 검색된 근거가 예측 후보를 얼마나 포함하는지 확인한다. 이는 분류 정확도 개선을 입증하는 지표가 아니라 검색 단계의 진단 지표다.

In [ ]:
top1 = evidence_df[evidence_df['rank'].eq(1)].copy()
top5 = evidence_df.copy()

def candidate_match(row):
    code = str(row['rule_hs_code']).strip()
    return bool(code) and code in row['candidate_hs_top3']

top5['direct_code_candidate_match'] = top5.apply(candidate_match, axis=1)
summary = pd.DataFrame([
    {'metric':'제품별 Top-1 검색 문서 수','value':len(top1)},
    {'metric':'검색 문서 중 HS 코드 직접 명시 비율','value':top5['rule_hs_code'].ne('').mean()},
    {'metric':'직접 HS 코드가 있는 문서 중 SVM Top-3 후보 일치 비율','value':top5.loc[top5['rule_hs_code'].ne(''),'direct_code_candidate_match'].mean() if top5['rule_hs_code'].ne('').any() else np.nan},
    {'metric':'검색된 지식 문서 고유 개수','value':top5['knowledge_id'].nunique()}
])
display(summary)
display(top5.groupby(['source_sheet','rule_type']).size().sort_values(ascending=False).rename('retrieved_count').reset_index())

## 5. 결과 저장

저장 파일은 제품별 모델 예측, 제품별 검색 근거, 지식베이스 검색 진단 요약이다. 검색 근거는 원문을 포함하므로 이후 사람이 규정 적용 여부를 검토할 수 있다.

In [ ]:
OUT = Path('/content/hs_knowledge_retrieval_results')
OUT.mkdir(parents=True, exist_ok=True)
pred_df.to_csv(OUT/'kb_model_predictions.csv', index=False, encoding='utf-8-sig')
evidence_df.to_csv(OUT/'kb_retrieval_evidence.csv', index=False, encoding='utf-8-sig')
summary.to_csv(OUT/'kb_retrieval_diagnostics.csv', index=False, encoding='utf-8-sig')
print('저장 위치:', OUT)
print('\n'.join(str(p) for p in sorted(OUT.glob('*.csv'))))

## 6. 다음 단계 판단

이 노트북은 검색 근거를 생성하는 RAG의 Retrieval 단계다. 검색된 문서가 실제 HS 분류 판단에 어떤 영향을 주는지는 아직 검증하지 않았다. 다음 단계에서는 검색 결과의 품질을 확인하고, 규칙별 적용범위·제외조건·소호체계를 구조화한 뒤, 규칙 기반 후보 재정렬 또는 LLM 기반 근거 설명을 별도 실험한다. 최종 성능 비교는 동일한 Fold와 동일한 평가 사례를 사용한다.